# 12 – Integrált PUNDIT esettanulmány: TK-101

**13. tanegység · javasolt idő: 2 × 50 perc**

## Tanulási célok

- a féléves módszereket egyetlen traceable workflow-ba integrálni
- qualitative → formal → dynamic → decision láncot végrehajtani
- eredmények és korlátok különválasztása
- önálló hallgatói projektet specifikálni

**PUNDIT kapcsolat.** Ez a tanegység a PUNDIT projekt (`2020-1.2.3-EUREKA-2022-00021`) eredményeinek **oktatási hasznosítását** támogatja; önmagában nem helyettesíti és nem igazolja a projekt eredeti műszaki vállalásainak teljesítését.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Projekt gyökér:", ROOT)


## 1. Integrációs térkép

**system → hazards → failure modes → PHA/FMEA/HAZOP → FTA → Bayesian evidence / Markov dynamics → Monte Carlo uncertainty → diagnostic update → RBM decision.**

A cél nem az, hogy minden módszert egyszerre használjunk, hanem hogy megértsük, melyik milyen kérdésre válaszol és hogyan ad át információt a következőnek.


## 2. Rendszermodell és hazard

A TK-101 modell strukturált YAML-ból indul. A representation simplified SysML-derived exchange format; nem natív SysML parser.


In [ ]:
import yaml
path=ROOT/'data'/'pundit_case'/'system_model.yaml'
with open(path,encoding='utf-8') as f: model=yaml.safe_load(f)
print(model.get('metadata',{})); print('components:',[c['id'] for c in model['components']]); print('hazards:',[h['id'] for h in model['hazards']])


## 3. Kvalitatív lánc

PHA: overflow/fire. FMEA: LT failed low, LSHH/XV fail-on-demand, operator miss. HAZOP: MORE LEVEL, NO EFFECTIVE SHUTDOWN. Ezekből származik az FTA struktúra.


In [ ]:
qual=pd.DataFrame([
 {'PHA':'Overflow','FMEA':'LT-101 failed low','HAZOP':'MORE LEVEL','formal':'FTA basic event'},
 {'PHA':'Overflow','FMEA':'LSHH fail on demand','HAZOP':'NO SHUTDOWN','formal':'FTA basic event'},
 {'PHA':'Overflow','FMEA':'XV fail on demand','HAZOP':'NO SHUTDOWN','formal':'FTA basic event'}]); qual


## 4. Model → FTA → baseline risk

A repository tesztelt modulja generálja a fault tree-t, majd minimal cut seteket és top-event probability-t számít.


In [ ]:
from safetycourse.model_io import load_yaml,validate_system_model,generate_fault_tree
from safetycourse.fta import top_event_probability,minimal_cut_sets
m=load_yaml(path); print(validate_system_model(m)); tree=generate_fault_tree(m,'OVERFLOW')
p0=top_event_probability(tree); print('baseline P(OVERFLOW)=',p0); print([sorted(x) for x in minimal_cut_sets(tree)])


## 5. Szenzoradat → evidencia

A szintetikus LT-101 residual alapján diagnosztikai score-t készítünk. A score→probability mapping oktatási modell és kalibrációt igényelne.


In [ ]:
df=pd.read_csv(ROOT/'data'/'pundit_case'/'sensor_data.csv'); r=df.lt101_measured_pct-df.true_level_pct
def sigmoid(x): return 1/(1+np.exp(-x))
p_sensor=.002+(.6-.002)*sigmoid(-((r+2.5)/.8))
plt.plot(df.time_min,p_sensor); plt.xlabel('time'); plt.ylabel('P(sensor fault | evidence)'); plt.grid(alpha=.3); plt.show()


## 6. Evidence → dynamic overflow risk

A statikus LT basic-event prior helyére a time-dependent posterior proxy kerül. Ez egy közvetlen szemléltetés arra, hogyan válhat a risk estimate adatérzékennyé.


In [ ]:
p_ch=.08; p_auto=1-(1-.01)*(1-.005); p_op=.04
p_dynamic=p_ch*p_auto*(1-(1-p_sensor)*(1-p_op))
plt.semilogy(df.time_min,p_dynamic); plt.axhline(p0,ls='--',label='baseline model'); plt.legend(); plt.xlabel('time'); plt.ylabel('P(overflow)'); plt.grid(alpha=.3); plt.show()


## 7. Markov dinamikai nézőpont

A state model a degraded állapotokban töltött időt és recovery transitionöket kezeli. Nem ugyanazt a kérdést teszi fel, mint az FTA; a két modell eredménye csak jól definiált mapping mellett kombinálható.


In [ ]:
from scipy.linalg import expm
states=['S0','S1','S2','S3','S4','S5']; Q=np.zeros((6,6))
def add(i,j,x): Q[i,j]+=x
add(0,1,.002); add(0,2,.001); add(0,3,.004); add(1,0,.08); add(1,3,.004); add(2,0,.05); add(2,3,.004); add(3,0,.2); add(3,4,.01); add(4,0,.1); add(4,5,.03)
for i in range(6): Q[i,i]=-Q[i].sum()
p=np.array([1,0,0,0,0,0.])@expm(Q*100); print(dict(zip(states,p))); print('P(S4 or S5)=',p[4]+p[5])


## 8. Monte Carlo uncertainty

A baseline basic-event probability-k bizonytalanságát Beta eloszlással szemléltetjük. A paraméterek szintetikusak.


In [ ]:
rng=np.random.default_rng(9); N=20_000
pch=rng.beta(8,92,N); pl=rng.beta(2,198,N); px=rng.beta(1,199,N); ps=rng.beta(4,196,N); po=rng.beta(8,192,N)
p_mc=pch*(1-(1-pl)*(1-px))*(1-(1-ps)*(1-po))
print('5/50/95%=',np.quantile(p_mc,[.05,.5,.95]))


## 9. Risk-based maintenance döntés

Ugyanaz a kérdés zárja a workflow-t, amellyel a projekt motivációja indult: **karbantartsuk az LT-101-et most, vagy várjunk a következő opportunity window-ig?**


In [ ]:
Cc=50_000_000; Cm=250_000; prod_now=800_000; prod_window=100_000; p_after=.001
cost_now=Cm+prod_now+p_after*Cc; current=float(p_dynamic.iloc[-1] if hasattr(p_dynamic,'iloc') else p_dynamic[-1]); cost_wait=Cm+prod_window+current*Cc
pd.DataFrame([{'option':'maintain now','expected_cost':cost_now},{'option':'wait','expected_cost':cost_wait}])


## 10. Traceability és PUNDIT educational exploitation

Dokumentálandó: source artifact, model version, synthetic/real data flag, assumptions, generated artifact, validation, chapter, project ID és `technical_deliverable_claim: false`. A repository `project/pundit_traceability.yml` ezt a szétválasztást szolgálja.


In [ ]:
trace=pd.read_csv(ROOT/'data'/'pundit_case'/'sensor_data.csv').head(1)
print('PUNDIT project: 2020-1.2.3-EUREKA-2022-00021')
print('use: educational_exploitation')
print('technical_deliverable_claim: false')


## 11. Féléves hallgatói projekt

Válassz egy safety-critical alrendszert. Minimum: system boundary; PHA/FMEA/HAZOP; egy formal risk model; legalább egy paraméterezhető Python notebook; uncertainty/assumption dokumentáció; egy maintenance/mitigation decision; traceability. Nem a modell mérete, hanem az érvelés minősége számít.


## Próbáld ki!

1. Válassz egy TK-101 basic eventet és tervezz adatforrást a probability kalibrálására.
2. Hasonlítsd össze, mit jelent a baseline FTA probability és a Markov S4+S5 probability; miért nem azonos mennyiség?
3. Adj common-cause eseményt a modelhez és dokumentáld a traceability-t.
4. Írj 5 mondatos decision memo-t a maintain now / wait döntésről, feltételezésekkel együtt.


## Összefoglalás

A kurzus végére a kockázatelemzés nem különálló módszerek listája, hanem traceable mérnöki lánc: **rendszermodell → veszély → formális modell → evidencia → dinamikus risk → döntés**. A PUNDIT kapcsolódás ennek oktatási hasznosítása, nem technikai deliverable-igazolás.
